# Refined Stage 3L - Moirai patch_size fairness re-run

The Moirai wrapper runs with `patch_size=32` throughout. Salesforce's reference example uses
`patch_size="auto"`. On 30-minute load a 32-step patch spans 16 hours — across the daily cycle
— which is the likely cause of Moirai's collapse on load (2.8x the best model) while it stays
competitive on wind (1.05x). A signal-specific deficit is the signature of a configuration
mismatch, not genuine weakness.

The thesis currently states **"Moirai never wins a Holm-corrected comparison."** That claim is
not safe until Moirai has been given its intended configuration. This notebook re-runs **only
Moirai** — Chronos and TimesFM are untouched — with both patch settings, head to head on the
exact windows Moirai was already scored on, and re-derives the model comparison.

**Cost:** Moirai is the cheapest model (~0.03 s/forecast). Two patch settings across the
scored windows is roughly 20–30 minutes on a GPU. Set `MAX_PER_CELL` below to subsample if you
want a faster first look.

In [1]:
import sys, os, time, warnings, importlib
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import refined_stage_common as C
importlib.reload(C)
REQUIRED = "2026.09.10-relmae"
assert getattr(C, "__version__", None) == REQUIRED, (
    f"stale refined_stage_common (got {getattr(C, '__version__', 'none')}, "
    f"need {REQUIRED}) -- restart the kernel and Run All")
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 45)
PALETTE = {"chronos": "#2563eb", "timesfm": "#059669", "moirai": "#d97706"}
print(f"common module {C.__version__} from {C.__file__}")

common module 2026.09.10-relmae from e:\Thesis\EnergyForcastModel\refined_stage_common.py


## 1. Reconstruct the windows Moirai was scored on

Each Moirai result row is reproduced exactly from the panel: with `origin_index = st` and
`context_steps = c`, the context is `vals[st-c:st]` and the target is `vals[st:st+h]`. The
stored `scale` and `snaive_mae` are reused, so relMAE is identical in definition to the rest of
the study.

In [2]:
DATA = C.get_data()
FILES = ["refined_stage3b_context_sweep.csv", "refined_stage3c_horizon_sweep.csv",
         "refined_stage3d_fixed_lead_time.csv"]
pool = pd.concat([pd.read_csv(f).assign(source=f.split("stage")[1][:2])
                  for f in FILES if os.path.exists(f)], ignore_index=True)

# unique Moirai window geometries (one Moirai row per window); keep the stored scale/benchmark
mo = pool[(pool.model == "moirai") & (~pool.degenerate)].dropna(subset=["snaive_mae"])
geo = (mo[["etype", "gran", "series_id", "origin_index", "context_steps", "horizon_steps",
           "scale", "snaive_mae", "relMAE"]]
       .drop_duplicates(subset=["etype", "gran", "series_id", "origin_index",
                                "context_steps", "horizon_steps"])
       .rename(columns={"relMAE": "relMAE_stored32"}).reset_index(drop=True))
print(f"{len(geo):,} unique Moirai windows to re-score")

MAX_PER_CELL = None       # e.g. 200 for a fast preview; None = every window (full, rigorous)
if MAX_PER_CELL:
    # collect indices per cell then .loc — version-proof (pandas 3.x drops group keys after apply)
    keep = []
    for _, g in geo.groupby(["etype", "gran", "context_steps", "horizon_steps"]):
        keep.extend(g.sample(min(len(g), MAX_PER_CELL), random_state=7).index)
    geo = geo.loc[keep].reset_index(drop=True)
    print(f"subsampled to {len(geo):,} windows (MAX_PER_CELL={MAX_PER_CELL})")

# reconstruct context/truth arrays once
ctxs, truths, ok = [], [], []
panels = {}
for r in geo.itertuples():
    key = (r.etype, r.gran)
    if key not in panels:
        panels[key] = C.panel(DATA, r.etype, r.gran)
    vals = panels[key][r.series_id]
    st, c, h = int(r.origin_index), int(r.context_steps), int(r.horizon_steps)
    if st - c < 0 or st + h > len(vals):
        ctxs.append(None); truths.append(None); ok.append(False); continue
    ctxs.append(np.asarray(vals[st - c:st], float))
    truths.append(np.asarray(vals[st:st + h], float)); ok.append(True)
geo["okflag"] = ok
print(f"reconstructed {sum(ok):,}/{len(geo):,} windows "
      f"({len(geo)-sum(ok)} unreconstructable, e.g. origin at series edge)")

loaded cached panel <- refined_panel_cache.pkl
  load   5 series, 230,736–232,272 points
  solar  5 series, 52,560–52,560 points
  wind   5 series, 434,876–434,876 points
21,129 unique Moirai windows to re-score
reconstructed 21,129/21,129 windows (0 unreconstructable, e.g. origin at series edge)


## 2. Score Moirai twice - patch_size 32 (reproduces the original) and "auto" (the fix)

Running `32` again on the reconstructed windows serves two purposes: it validates that the
reconstruction reproduces the stored scores (section 3), and it gives a clean internal
head-to-head against `auto` in a single session and code path.

In [3]:
def score_moirai(patch, ctx_list, truth_list, geo_df):
    """Return relMAE per window for one patch_size, on the reconstructed windows."""
    mdl = C.MoiraiFM(patch_size=patch)
    mdl.max_cached = 64                        # many (horizon, context) pairs; share one module
    max_ctx = int(geo_df.loc[geo_df.okflag, "context_steps"].max())
    max_hor = int(geo_df.loc[geo_df.okflag, "horizon_steps"].max())
    mdl.configure(max_ctx, max_hor); mdl.warmup()
    out = np.full(len(geo_df), np.nan); t0 = time.perf_counter(); n = 0
    try:
        from tqdm.auto import tqdm
    except Exception:
        def tqdm(x, **k): return x
    for i, (row, ctx, tru) in enumerate(zip(geo_df.itertuples(), ctx_list, truth_list)):
        if not row.okflag:
            continue
        try:
            p = np.asarray(mdl.predict(ctx, int(row.horizon_steps)), float)[:int(row.horizon_steps)]
            if p.shape[0] == int(row.horizon_steps) and np.all(np.isfinite(p)):
                out[i] = C.metrics(tru, p, row.scale, row.snaive_mae)["relMAE"]
                n += 1
        except Exception:
            pass
        if (i + 1) % 2000 == 0:
            print(f"    patch={patch}: {i+1:,}/{len(geo_df):,}  ({time.perf_counter()-t0:.0f}s)",
                  flush=True)
    mdl.free()
    print(f"  patch={patch}: scored {n:,} windows in {time.perf_counter()-t0:.0f}s")
    return out

print("scoring patch_size=32 (reproduction) ...", flush=True)
geo["relMAE_32"] = score_moirai(32, ctxs, truths, geo)
print("\nscoring patch_size='auto' (the fix) ...", flush=True)
geo["relMAE_auto"] = score_moirai("auto", ctxs, truths, geo)
geo.to_csv("refined_stage3l_moirai_patch.csv", index=False)
print("\nsaved refined_stage3l_moirai_patch.csv")

scoring patch_size=32 (reproduction) ...
    patch=32: 2,000/21,129  (110s)
    patch=32: 4,000/21,129  (177s)
    patch=32: 6,000/21,129  (251s)
    patch=32: 8,000/21,129  (303s)
    patch=32: 10,000/21,129  (354s)
    patch=32: 12,000/21,129  (403s)
    patch=32: 14,000/21,129  (452s)
    patch=32: 16,000/21,129  (501s)
    patch=32: 18,000/21,129  (554s)
    patch=32: 20,000/21,129  (625s)
  patch=32: scored 21,129 windows in 673s

scoring patch_size='auto' (the fix) ...
    patch=auto: 2,000/21,129  (4s)
    patch=auto: 4,000/21,129  (8s)
    patch=auto: 6,000/21,129  (12s)
    patch=auto: 8,000/21,129  (17s)
    patch=auto: 10,000/21,129  (22s)
    patch=auto: 12,000/21,129  (26s)
    patch=auto: 14,000/21,129  (30s)
    patch=auto: 16,000/21,129  (34s)
    patch=auto: 18,000/21,129  (38s)
    patch=auto: 20,000/21,129  (42s)
  patch=auto: scored 0 windows in 44s

saved refined_stage3l_moirai_patch.csv


## 3. Validation - does the reconstruction reproduce the original Moirai run?

`relMAE_32` (re-scored here) should match `relMAE_stored32` (from the sweep files). Moirai is
seeded, so the two should agree to numerical precision. A large discrepancy means the
reconstruction is not faithful and the comparison below cannot be trusted.

In [4]:
v = geo.dropna(subset=["relMAE_32", "relMAE_stored32"])
diff = (v.relMAE_32 - v.relMAE_stored32).abs()
print(f"windows compared: {len(v):,}")
print(f"max |relMAE_32 - relMAE_stored32| = {diff.max():.3e}")
print(f"median abs diff                   = {diff.median():.3e}")
print(f"agree within 1e-6: {(diff < 1e-6).mean():.1%}")
if diff.max() > 1e-3:
    print("\n!! reconstruction does NOT reproduce the stored run to precision — investigate")
    print("   (a small residual is acceptable if Moirai sampling differs across library versions)")
else:
    print("\nreconstruction faithful: the head-to-head below is on identical windows.")

windows compared: 20,351
max |relMAE_32 - relMAE_stored32| = 9.541e-10
median abs diff                   = 0.000e+00
agree within 1e-6: 100.0%

reconstruction faithful: the head-to-head below is on identical windows.


## 4. Does `auto` beat `32`? Per cell.

In [5]:
cl = geo.dropna(subset=["relMAE_32", "relMAE_auto"])
cell = (cl.groupby(["etype", "gran"])
          .agg(relMAE_32=("relMAE_32", "median"),
               relMAE_auto=("relMAE_auto", "median"),
               n=("relMAE_auto", "size")).reset_index())
cell["improvement_%"] = (100 * (cell.relMAE_32 - cell.relMAE_auto) / cell.relMAE_32).round(1)
cell["auto_better"] = cell.relMAE_auto < cell.relMAE_32
print(cell.round(3).to_string(index=False))
print(f"\nauto improves {int(cell.auto_better.sum())} of {len(cell)} cells; "
      f"median improvement {cell['improvement_%'].median():.1f}%")

from scipy.stats import wilcoxon
print("\npaired Wilcoxon per cell (auto vs 32, on shared windows):")
for (et, gk), g in cl.groupby(["etype", "gran"]):
    a, b = g.relMAE_auto.values, g.relMAE_32.values
    try:
        p = wilcoxon(a, b).pvalue
    except ValueError:
        p = 1.0
    tag = "auto better" if np.median(a) < np.median(b) else "32 better"
    print(f"  {et:5s}/{gk:6s}  n={len(g):5d}  p={p:.2e}  {tag}")
cell.to_csv("refined_stage3l_cell_comparison.csv", index=False)

Empty DataFrame
Columns: [etype, gran, relMAE_32, relMAE_auto, n, improvement_%, auto_better]
Index: []

auto improves 0 of 0 cells; median improvement nan%

paired Wilcoxon per cell (auto vs 32, on shared windows):


## 5. The claim under test - does Moirai now win any cell?

Substitute Moirai's `auto` score for its `32` score and re-derive the per-cell best model
against the stored Chronos and TimesFM medians. If Moirai now wins somewhere, the thesis
sentence "Moirai never wins" must be revised; if not, the sentence stands and is now defensible
because Moirai was fairly configured.

In [6]:
ct = (pool[pool.model.isin(["chronos", "timesfm"]) & ~pool.degenerate]
        .dropna(subset=["relMAE"])
        .groupby(["etype", "gran", "model"]).relMAE.median().reset_index())
ct = ct.pivot_table(index=["etype", "gran"], columns="model", values="relMAE")

mo_auto = cl.groupby(["etype", "gran"]).relMAE_auto.median().rename("moirai_auto")
mo_32   = cl.groupby(["etype", "gran"]).relMAE_32.median().rename("moirai_32")
board = ct.join(mo_auto).join(mo_32).reset_index()

def winner(row, moirai_col):
    cand = {"chronos": row.get("chronos"), "timesfm": row.get("timesfm"),
            "moirai": row.get(moirai_col)}
    cand = {k: v for k, v in cand.items() if pd.notna(v)}
    return min(cand, key=cand.get) if cand else None

board["best_with_moirai32"]   = board.apply(lambda r: winner(r, "moirai_32"), axis=1)
board["best_with_moiraiauto"] = board.apply(lambda r: winner(r, "moirai_auto"), axis=1)
board["changed"] = board.best_with_moirai32 != board.best_with_moiraiauto
print(board.round(3).to_string(index=False))
nwin = int((board.best_with_moiraiauto == "moirai").sum())
print(f"\ncells Moirai wins with patch_size=32:   "
      f"{int((board.best_with_moirai32=='moirai').sum())} of {len(board)}")
print(f"cells Moirai wins with patch_size=auto: {nwin} of {len(board)}")
if nwin > 0:
    print("\n>>> 'Moirai never wins' must be REVISED — it wins with correct configuration:")
    print(board[board.best_with_moiraiauto == "moirai"][["etype", "gran", "moirai_auto",
          "chronos", "timesfm"]].round(3).to_string(index=False))
else:
    print("\n>>> 'Moirai never wins' STANDS, and is now defensible: it holds even with the")
    print("    recommended patch_size='auto'.")
board.to_csv("refined_stage3l_model_board.csv", index=False)

etype   gran  chronos  timesfm  moirai_auto  moirai_32 best_with_moirai32 best_with_moiraiauto  changed
 load     1D    0.795    0.498          NaN        NaN            timesfm              timesfm    False
 load     1h    0.630    0.665          NaN        NaN            chronos              chronos    False
 load native    0.725    0.473          NaN        NaN            timesfm              timesfm    False
solar     1D    0.769    0.752          NaN        NaN            timesfm              timesfm    False
solar     1h    0.891    1.016          NaN        NaN            chronos              chronos    False
solar native    1.125    1.023          NaN        NaN            timesfm              timesfm    False
 wind     1D    0.993    0.952          NaN        NaN            timesfm              timesfm    False
 wind     1h    1.158    0.967          NaN        NaN            timesfm              timesfm    False
 wind native    0.616    0.589          NaN        NaN          

## 6. What to report

- **Section 3** must pass (reconstruction faithful) for anything below it to count. Quote the
  max discrepancy.
- **Section 4** is the direct result: how much `patch_size` alone changes Moirai, per cell.
  Load native is the cell to watch — that is where the 32-step patch was most misaligned with
  the daily cycle.
- **Section 5** decides the sentence in the thesis. Either Moirai now wins somewhere (revise the
  claim, and report which cells), or it does not (the claim stands and is now robust to the
  configuration objection).
- Either outcome strengthens the thesis: it removes an obvious "you mis-configured the model"
  criticism by settling it with evidence. Note in the methodology that all foundation models
  were run with their reference-recommended settings.
- If `auto` changes Moirai materially, the Stage 3E model-significance tests that involve Moirai
  should be re-run with the `auto` scores before the final model-ranking claim is quoted.